# Virtual Mouse / Air Canvas - Phase 5

## Air drawing on a full-desktop overlay

Draw in the air and the strokes appear **on top of your whole screen** - over
VS Code, the browser, anything. The drawing surface is a transparent,
always-on-top, click-through window covering the desktop.

| Gesture | Action |
|---------|--------|
| **Index finger pointing** | pen DOWN - draws |
| **Pinch** (index+thumb) | pen UP - move without drawing |
| **Open palm** | clear the canvas |
| **Fist** | exit the overlay |

---
### How this works (it's different from the camera window)
- A **PyQt5** window covers the full screen. It is transparent (you see your
  desktop through it), stays on top of every other window, and is
  **click-through** - clicks pass to VS Code underneath.
- The MediaPipe hand tracking runs on a **background thread** and sends the
  fingertip position to the overlay, which paints the strokes.

### Important notes before running
- This needs a **real display** - it cannot run in Colab or a headless server.
- Run this as a **`.py` file** (right-click -> Run Python File) rather than
  cell-by-cell. PyQt's event loop and Jupyter's event loop conflict; running
  as a script is the reliable way. The notebook cell below writes the script
  for you, then you run it from the terminal.
- A small **camera preview window** also opens so you can see your hand and the
  status. Press **`q`** there, make a **fist**, or hit **Esc** to quit.

## 1. Install PyQt5 (run once)
Uncomment, run once, then comment it back.

In [1]:
# !pip install PyQt5

## 2. Write the overlay script

This cell writes `air_canvas.py` to disk. Because PyQt and Jupyter event loops
clash, you then run that file from the terminal:

```
python air_canvas.py
```

In [2]:
script = r'''"""
Air Canvas - full-desktop transparent drawing overlay.
Index finger = draw, pinch = pen up, open palm = clear, fist = exit.
Run:  python air_canvas.py
"""
import sys
import math
import time
import platform
import threading
from collections import deque

import cv2
import numpy as np
import mediapipe as mp
from PyQt5 import QtCore, QtWidgets, QtGui


# ----------------------------------------------------------------------
# Hand tracking (runs on a background thread, pushes points to the overlay)
# ----------------------------------------------------------------------
WRIST = 0
THUMB_TIP, INDEX_TIP, MIDDLE_TIP = 4, 8, 12
INDEX_MCP = 5
TIPS = [4, 8, 12, 16, 20]
PIPS = [3, 6, 10, 14, 18]

CLICK_ON = 0.18          # index-thumb pinch threshold (pen up)
DEBOUNCE = 3

# interaction box (fraction of camera frame) mapped to the full screen
ACTIVE_LEFT, ACTIVE_RIGHT = 0.25, 0.75
ACTIVE_TOP, ACTIVE_BOTTOM = 0.20, 0.70
ALPHA = 0.4              # smoothing


def fingers_up(lm):
    f = [1 if lm[THUMB_TIP].x < lm[THUMB_TIP - 1].x else 0]
    for tip, pip in zip(TIPS[1:], PIPS[1:]):
        f.append(1 if lm[tip].y < lm[pip].y else 0)
    return f


def hand_size(lm):
    return math.hypot(lm[INDEX_MCP].x - lm[WRIST].x,
                      lm[INDEX_MCP].y - lm[WRIST].y) + 1e-6


def pinch_ratio(lm):
    d = math.hypot(lm[INDEX_TIP].x - lm[THUMB_TIP].x,
                   lm[INDEX_TIP].y - lm[THUMB_TIP].y)
    return d / hand_size(lm)


class HandThread(threading.Thread):
    """Captures camera, tracks hand, emits (x, y, pen_down, command)."""
    def __init__(self, screen_w, screen_h, on_point, on_command):
        super().__init__(daemon=True)
        self.screen_w = screen_w
        self.screen_h = screen_h
        self.on_point = on_point        # callback(x, y, pen_down)
        self.on_command = on_command    # callback("clear" | "exit")
        self.running = True
        self.ema_x = None
        self.ema_y = None
        self.pen_candidate = False
        self.pen_count = 0
        self.pen_down = False
        self.pinching = False

    def _smooth(self, x, y):
        if self.ema_x is None:
            self.ema_x, self.ema_y = x, y
        else:
            self.ema_x = ALPHA * x + (1 - ALPHA) * self.ema_x
            self.ema_y = ALPHA * y + (1 - ALPHA) * self.ema_y
        return self.ema_x, self.ema_y

    def run(self):
        if platform.system() == "Windows":
            cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
        else:
            cap = cv2.VideoCapture(0)
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

        hands = mp.solutions.hands.Hands(
            static_image_mode=False, max_num_hands=1,
            min_detection_confidence=0.7, min_tracking_confidence=0.5)
        draw_util = mp.solutions.drawing_utils

        while self.running:
            ok, frame = cap.read()
            if not ok:
                continue
            frame = cv2.flip(frame, 1)
            h, w = frame.shape[:2]
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            rgb.flags.writeable = False
            res = hands.process(rgb)
            rgb.flags.writeable = True

            status = "no hand"
            if res.multi_hand_landmarks:
                hand = res.multi_hand_landmarks[0]
                draw_util.draw_landmarks(
                    frame, hand, mp.solutions.hands.HAND_CONNECTIONS)
                lm = hand.landmark
                f = fingers_up(lm)
                total = sum(f)

                if total == 0:
                    self.on_command("exit")
                    status = "FIST -> exit"
                elif total == 5:
                    self.on_command("clear")
                    status = "PALM -> clear"
                else:
                    ratio = pinch_ratio(lm)
                    # Debounce the pinch: only change the confirmed state after a
                    # new reading persists for DEBOUNCE frames. A single noisy
                    # frame can never flip it, so the pen line stays intact.
                    raw_pinch = ratio < CLICK_ON
                    if raw_pinch != self.pen_candidate:
                        self.pen_candidate = raw_pinch
                        self.pen_count = 1
                    else:
                        self.pen_count += 1
                        if self.pen_count >= DEBOUNCE:
                            self.pinching = self.pen_candidate

                    # Pen DOWN when the index finger points and we are NOT pinching.
                    index_pointing = f[1] == 1
                    self.pen_down = index_pointing and not self.pinching

                    tip = lm[INDEX_TIP]
                    sx = np.interp(tip.x, (ACTIVE_LEFT, ACTIVE_RIGHT), (0, self.screen_w))
                    sy = np.interp(tip.y, (ACTIVE_TOP, ACTIVE_BOTTOM), (0, self.screen_h))
                    sx, sy = self._smooth(sx, sy)
                    sx = float(np.clip(sx, 0, self.screen_w - 1))
                    sy = float(np.clip(sy, 0, self.screen_h - 1))
                    self.on_point(sx, sy, self.pen_down)
                    status = "DRAW" if self.pen_down else "pen up"

            cv2.putText(frame, status, (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
            cv2.imshow("Air Canvas - hand view (q to quit)", frame)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                self.on_command("exit")
                break

        cap.release()
        cv2.destroyAllWindows()
        hands.close()

    def stop(self):
        self.running = False


# ----------------------------------------------------------------------
# Transparent, always-on-top, click-through overlay
# ----------------------------------------------------------------------
class Overlay(QtWidgets.QWidget):
    point_signal = QtCore.pyqtSignal(float, float, bool)
    command_signal = QtCore.pyqtSignal(str)

    def __init__(self):
        super().__init__()
        self.setWindowFlags(
            QtCore.Qt.FramelessWindowHint |
            QtCore.Qt.WindowStaysOnTopHint |
            QtCore.Qt.Tool)
        self.setAttribute(QtCore.Qt.WA_TranslucentBackground, True)
        self.setAttribute(QtCore.Qt.WA_TransparentForMouseEvents, True)

        screen = QtWidgets.QApplication.primaryScreen().geometry()
        self.setGeometry(screen)
        self.screen_w = screen.width()
        self.screen_h = screen.height()

        self.strokes = []          # list of polylines; each is list of QPointF
        self.current = None
        self.last_pen = False

        self.point_signal.connect(self.add_point)
        self.command_signal.connect(self.handle_command)

        # repaint timer
        self.timer = QtCore.QTimer(self)
        self.timer.timeout.connect(self.update)
        self.timer.start(16)       # ~60 fps

        self.showFullScreen()

    @QtCore.pyqtSlot(float, float, bool)
    def add_point(self, x, y, pen_down):
        if pen_down:
            # Start a new stroke whenever there is no active one (covers the
            # first point, the point right after a pen-up, and after a clear).
            if not self.last_pen or self.current is None:
                self.current = [QtCore.QPointF(x, y)]
                self.strokes.append(self.current)
            else:
                self.current.append(QtCore.QPointF(x, y))
        else:
            # pen up: end the current stroke so the next down starts fresh
            self.current = None
        self.last_pen = pen_down

    @QtCore.pyqtSlot(str)
    def handle_command(self, cmd):
        if cmd == "clear":
            self.strokes = []
            self.current = None
        elif cmd == "exit":
            QtWidgets.QApplication.quit()

    def paintEvent(self, event):
        painter = QtGui.QPainter(self)
        painter.setRenderHint(QtGui.QPainter.Antialiasing, True)
        pen = QtGui.QPen(QtGui.QColor(0, 220, 255), 5,
                         QtCore.Qt.SolidLine, QtCore.Qt.RoundCap, QtCore.Qt.RoundJoin)
        painter.setPen(pen)
        for stroke in self.strokes:
            if len(stroke) > 1:
                painter.drawPolyline(QtGui.QPolygonF(stroke))
        # also handle Esc to quit
    def keyPressEvent(self, event):
        if event.key() == QtCore.Qt.Key_Escape:
            QtWidgets.QApplication.quit()


def main():
    app = QtWidgets.QApplication(sys.argv)
    overlay = Overlay()
    hand = HandThread(
        overlay.screen_w, overlay.screen_h,
        on_point=lambda x, y, p: overlay.point_signal.emit(x, y, p),
        on_command=lambda c: overlay.command_signal.emit(c))
    hand.start()
    app.exec_()
    hand.stop()
    time.sleep(0.3)


if __name__ == "__main__":
    main()
'''

with open("air_canvas.py", "w") as f:
    f.write(script)
print("Wrote air_canvas.py")
print("Now run it from the terminal:  python air_canvas.py")

Wrote air_canvas.py
Now run it from the terminal:  python air_canvas.py


## 3. Run it

In the VS Code terminal (with your venv active):

```
python air_canvas.py
```

Two things appear: a small camera window showing your hand, and an invisible
full-screen drawing layer. Point your index finger to draw on your whole
desktop; pinch to lift the pen; open palm to clear; make a fist (or press `q`
in the camera window, or `Esc`) to exit.

### If something goes wrong
- **Nothing draws / can't see lines:** the overlay is transparent by design -
  lines only appear where you have drawn. Try a big slow stroke first.
- **Can't exit:** make a fist, or click the camera window and press `q`, or
  `Esc`. Worst case, `Ctrl+C` in the terminal.
- **Lines lag behind finger:** raise `ALPHA` in the script (e.g. 0.5-0.6).
- **Pen draws when you don't want / won't lift:** adjust `CLICK_ON` (your pinch
  measured ~0.10 touching, so 0.18 should lift cleanly).

## Notes & next steps
- This is the hardest component in the project - a real desktop overlay with
  live hand input. Getting it running is a genuine milestone.
- Possible upgrades: multiple pen colours (gesture to cycle), adjustable
  thickness, an eraser gesture, save the canvas to a PNG.
- **Phase 6** (optional) is the C++ injection module, only worth it if you ever
  measure that actions feel laggy. **Phase 7** is handwriting recognition -
  turning drawn strokes into typed text.